### Import Dependencies

In [118]:
from qdrant_client import QdrantClient
import instructor
from pydantic import BaseModel, Field
from typing import Optional
from typing import List
from instructor import Mode
import yaml
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from langchain_community.chat_models import ChatOllama
from ragas.llms import LangchainLLMWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch
from qdrant_client.models import Document,Prefetch,FusionQuery
import yaml
from jinja2 import Template
from bs4 import BeautifulSoup


C:\Users\SURYA ER\AppData\Local\Temp\ipykernel_64292\3248950423.py:10: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\SURYA ER\AppData\Local\Temp\ipykernel_64292\3248950423.py:10: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\SURYA ER\AppData\Local\Temp\ipykernel_64292\3248950423.py:10: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\SURYA ER\AppData\Local\Te

In [135]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-mpnet-base-v2", device=device)
reranker = CrossEncoder("BAAI/bge-reranker-base")
qdrant_client = QdrantClient(url="http://localhost:6333")
client = instructor.from_provider(
    "ollama/llama3",
    base_url="http://localhost:11434/v1",
    mode=Mode.JSON   
    )
ragas_llm = ChatOllama(
    model="llama3",
    temperature=0
)
ragas_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

C:\Users\SURYA ER\AppData\Local\Temp\ipykernel_64292\4050724877.py:10: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  ragas_llm = ChatOllama(
C:\Users\SURYA ER\AppData\Local\Temp\ipykernel_64292\4050724877.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  ragas_embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SURYA ER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [110]:
class SyntheticDataFields(BaseModel):
    question: str = Field(description="Suggested question")
    context_ids: list[str] = Field(description = "id of the chunk that could be used to answer the question")
    answer_example: str= Field(description="Suggested answer rounded in the context")
    answer: str= Field(description="Reasoning why the question could be answered with the chunks")


class SyntheticData(BaseModel):
    response: list[SyntheticDataFields] = Field(description="The list of SyntheticDataFields")

class RAGUsedContext(BaseModel):
    id: str = Field(description="The appid of the item used to answer the question")
    description: str = Field(description = "Short description of the item used to answer the question")


class RAGGenerationResponse(BaseModel):
    answer:str = Field(description="The answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [91]:
def rerank(query, docs,top_k=3):

    if not docs:
        return []

    pairs = [(query, d["description"]) for d in docs]

    scores = reranker.predict(pairs)

    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)

    ranked_docs = sorted(
        docs,
        key=lambda d: d["rerank_score"],
        reverse=True
    )
    return ranked_docs[:top_k] if top_k else ranked_docs

In [73]:
def get_embeddings(text):
    
    embedded_text = embedding_model.encode(inputs=text, normalize_embeddings = True, convert_to_numpy = True)
    return embedded_text.tolist()

In [99]:
def generate_answer(prompt):
    response = client.create(
    model="llama3",
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    response_model=RAGGenerationResponse
    )

    return response.model_dump()

In [93]:
def process_context(docs):
    formatted = ""

    for context in docs:
        formatted += f"""
ID: {context["appid"]}
Name: {context["name"]}
Genre: {context["genre"]}
Score: {context["similarity_score"]}
Description: {context["description"]}
---
"""
    return formatted

In [131]:
llm_prompt= None
with open("./prompts/generic_prompt.yaml", "r") as yaml_file:
        llm_prompt = yaml.safe_load(yaml_file)["prompts"]["retrieval_generation"]

In [120]:
def clean_text(text):
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)

In [ ]:
def rag_pipeline(query, limit = 10):


    embedded_query = get_embeddings(query)
    results = qdrant_client.query_points(collection_name="steam-data-collection-hybrid-search", limit=limit, prefetch=[

        Prefetch(
            query=embedded_query,
            using="dense",
            limit=limit
        ),
        Prefetch(
            query=Document(text=query,model="qdrant/bm25"),
            using="sparse",
            limit=limit
        )
    ], query = FusionQuery(fusion="rrf"))

    retrieved_docs = []

    for result in results.points:
        payload = result.payload
        ### I find this as a huge mistake, whatever data we embbed and store in V DB as vectors it's better to store them in payload as well :)
        clean_description = clean_text(payload["name"] + " " + payload['detailed_description'] + " " +  payload['about_the_game'] + " " +payload['short_description'])
        retrieved_docs.append ({
            "appid": payload["appid"],
            "description": clean_description[:300],
            "similarity_score":result.score,
            "genre": payload["genres"],
            "name": payload["name"]
        })


    reranked_result = rerank(query,retrieved_docs)
    processed_context = process_context(reranked_result)
    
    prompt_template = Template(llm_prompt)
    processed_prompt = prompt_template.render(processed_context = processed_context, question = query)
    answer = generate_answer(processed_prompt)

    retrieved_context = [doc["description"] for doc in  reranked_result]

    return answer["answer"], retrieved_context


In [125]:
result = rag_pipeline("suggest me fps multiplayer games")

In [126]:
print(result)

("If you're looking for a multiplayer experience in First-Person Shooter (FPS) games, both Aliens vs. Predator and Joint Operations: Combined Arms Gold offer unique multiplayer experiences.", ["Aliens vs. Predator™ Bringing the legendary war between two of science-fiction's most popular characters to FPS fans, AvP delivers three outstanding single player campaigns and provides untold hours of unique 3-way multiplayer gaming. Experience distinctly new and thrilling first person gameplay as ", 'Joint Operations: Combined Arms Gold The most ferocious online conflict of the 21st century is now available in its most definitive form: introducing Joint Operations: Combined Arms Gold, the largest multiplayer FPS collection ever. With the award-winning Joint Operations: Typhoon Rising and the bes', 'Painkiller Overdose The critically-acclaimed and award-winning FPS franchise is back! Packed with tons of fast-paced, adrenaline-fueled single player and multiplayer action, Painkiller Overdose brin

In [57]:
system_prompt = """
I am building a RAG application. i have a collection of 100 chunks of text.
The RAG application will act as as game shopping assistant that can answer questions about a specific game.
I wil provide all of the available games to wyou with IDS of each chunk.
I want you to come up with 30 quesetions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them. 
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that coulld use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

Return ONLY valid JSON.

{
  "response": [
    {
      "question": "string",
      "context_ids": ["string"],
      "answer_example": "string",
      "answer": "string"
    }
  ]
}


Rules:
- No extra keys
- No explanations
- No schema
- No comments
- No text outside JSON
- Use exactly these field names
- No HTML elements tags or references

IMPORTANT:
- context_ids MUST be copied EXACTLY from the provided Chunk ID
- DO NOT invent new IDs
- DO NOT use titles as IDs
- ONLY use the provided Chunk ID values
"""

In [54]:
all_points = qdrant_client.scroll(collection_name = "steam-data-collection-hybrid-search", limit=100,offset=None,with_payload=True,with_vectors=False)

In [55]:
all_points[0][0].payload["name"]

'Counter-Strike'

In [17]:
### we are just formatting the description how we embedded as vector in our qdrant collection
def format_description(data):

    return data["name"] + " " + data['detailed_description'] + " " +  data['about_the_game'] + " " + data['short_description'] 

In [22]:
data_for_synthesizing  = [{"id": point.payload["appid"], "description":format_description(point.payload)}  for point in all_points[0]]

In [25]:
len(data_for_synthesizing)

100

In [46]:
user_prompt = f"""
Here is the list of Id and description
{data_for_synthesizing}
"""

In [58]:
resp = client.create(
    model="llama3",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt,
        }
    ],
    response_model=SyntheticData,
)
print(resp.model_dump_json())

{"response":[{"question":"What is the main genre of Cossacks: European Wars?","context_ids":["Cossacks: European Wars Cossacks: European Wars is a historical real-time strategy based on events of the 16th through the 18th centuries in Europe, when nations and states were created and demolished, and wars shed seas of blood. There are 16 nations or regions in Cossacks: Algeria, Austria, England, France, the Netherlands, Piemonte, Poland, Portugal, Prussia, Russia, Saxony, Spain, Sweden, Turkey, Ukraine, and Venice. Each has its own original graphics, economic and technical development peculiarities, military advantages and drawbacks, and unique units and technologies, providing vast choices of tactics and strategy in war against any enemy. Thus, England is the mightiest sea power, Austria has powerful light and heavy cavalry, and Cossacks are the pride of the Ukrainian army."],"answer_example":"It's a real-time strategy game based on historical events.","answer":"The question could be an

In [66]:
synthetic_data = resp.model_dump()["response"]

#### OBSERVATION - well we received the intended output but the context_ids - DAMN! this is odd and hilarious anyhow moving on since RAGAS don't need IDs anyhow on production we need to do this even better :)

In [77]:
len(synthetic_data)

19

In [132]:
eval_data = []

for item in synthetic_data:

    answer, contexts = rag_pipeline(item["question"])
    eval_data.append({
        "question": item["question"],
        "ground_truth": item["answer_example"],
        "answer": answer,
        "contexts": contexts
    })

In [137]:
len(eval_data)

19

In [136]:
dataset = Dataset.from_list(eval_data)
result = evaluate(
     dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)
print(result)

Evaluating:   0%|          | 0/57 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\SURYA ER\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002C2DAF3D100> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-891' coro=<_async_in_context.<locals>.run_in_context() done, defined at d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-894' coro=<Kernel.shell_main() running at d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
d:\learning\my-rag-pipeline-master\.venv\Lib\site-packages\r

{'faithfulness': 0.6667, 'answer_relevancy': 0.7770, 'context_recall': 0.7917}


In [ ]:
result

### output: {'faithfulness': 0.6667, 'answer_relevancy': 0.7770, 'context_recall': 0.7917}

{'faithfulness': 0.6667, 'answer_relevancy': 0.7770, 'context_recall': 0.7917}